# English Premier League VAR Analysis
## Part 1 - Data Acquisition

### Overview
This notebook scrapes VAR (Video Assistant Referee) decision data from ESPN articles covering
Premier League seasons from 2019/2020 to 2024/2025. The data includes net VAR impact scores,
overturned decisions, goals awarded/disallowed, and penalty statistics for all 20 EPL clubs.

### Technical Approach
- **Selenium WebDriver**: Required because ESPN uses React (client-side rendering). Static HTTP 
  requests only return the HTML shell without the dynamically-loaded article content.
- **Fresh driver per page**: ESPN's SPA can have session state issues when reusing drivers across
  multiple page loads, causing intermittent failures.
- **Regex pattern matching**: Team names are extracted using `\s([+-]\d+|0)$` to match the 
  "Team Name +N" format while avoiding false positives like "Ligue 1".

### Output
- `data/var_decisions_all_seasons.csv`: Combined dataset with ~120 rows (20 teams × 6 seasons)

### Data Sources

#### ESPN VAR Analysis Articles (Primary Source)
ESPN publishes end-of-season VAR impact analysis articles. These are JavaScript-rendered React pages.

| Season | URL | Notes |
|--------|-----|-------|
| 2019/2020 | [espn.com/37575919](https://www.espn.com/soccer/story/_/id/37575919/how-var-decisions-affected-every-premier-league-club) | First full VAR season |
| 2020/2021 | [espn.co.uk/37587139](https://www.espn.co.uk/football/story/_/id/37587139/how-var-decisions-affected-every-premier-league-club-2020-21) | COVID-affected season |
| 2021/2022 | [espn.co.uk/37619801](https://www.espn.co.uk/football/story/_/id/37619801/how-var-decisions-affected-every-premier-league-club-2021-22) | |
| 2022/2023 | [espn.co.uk/37631044](https://www.espn.co.uk/football/story/_/id/37631044/how-var-decisions-affected-every-premier-league-club-2022-23) | |
| 2023/2024 | [espn.com/38196464](https://www.espn.com/soccer/story/_/id/38196464/how-var-decisions-affect-premier-league-club-2023-24) | |
| 2024/2025 | [espn.co.uk/40894476](https://www.espn.co.uk/football/story/_/id/40894476/how-var-decisions-affect-premier-league-club-2024-25) | Latest available |

#### EPL Standings (Secondary Source)
- Historical standings are hardcoded since Eurosport/TNT Sports changed their page structure.
- For current season standings, consider using the [football-data.org API](https://www.football-data.org/).


In [1]:
"""
ESPN VAR Data Scraper
=====================
Scrapes VAR decision statistics from ESPN's Premier League analysis articles.

Why Selenium?
-------------
ESPN uses React for client-side rendering. A simple requests.get() only returns the 
HTML shell (<div id="root"></div>) without the actual article content. Selenium renders
the JavaScript first, giving us access to the full DOM.

Architecture Decision: Fresh Driver Per Page
---------------------------------------------
ESPN's React app can have session state issues (cookies, localStorage, React state) that
cause subsequent page loads to fail or return incomplete data. Creating a fresh WebDriver
instance for each URL is slower (~10s overhead) but ensures consistent, reliable scraping.

Alternative approaches considered:
- driver.delete_all_cookies() between pages: Doesn't clear React state
- Page refresh loops: Unreliable timing
- Explicit waits for specific elements: ESPN's selectors change across article versions
"""

# =============================================================================
# IMPORTS
# =============================================================================
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# =============================================================================
# WEBDRIVER MANAGER (Optional but Recommended)
# =============================================================================
# webdriver-manager handles ChromeDriver version matching automatically.
# Without it, you need to manually download the correct ChromeDriver version
# that matches your installed Chrome browser version.
try:
    from webdriver_manager.chrome import ChromeDriverManager
    USE_WEBDRIVER_MANAGER = True
except ImportError:
    USE_WEBDRIVER_MANAGER = False
    print("⚠️ webdriver-manager not installed. Install with: pip install webdriver-manager")

# =============================================================================
# WEBDRIVER INITIALIZATION
# =============================================================================
def init_driver(headless=True):
    """
    Initialize a Chrome WebDriver instance with production-ready settings.
    
    Args:
        headless: Run browser without GUI. Set False for debugging to see what's happening.
    
    Returns:
        webdriver.Chrome: Configured Chrome driver instance
    
    Chrome Options Explained:
        --headless: No GUI, required for server/CI environments
        --no-sandbox: Bypass OS security model (needed in Docker/restricted environments)
        --disable-dev-shm-usage: Overcome limited /dev/shm in Docker
        --disable-gpu: Disable GPU hardware acceleration (reduces issues in headless mode)
        --window-size: Set viewport size (affects responsive content rendering)
        user-agent: Mimic real browser to avoid bot detection
    """
    options = Options()
    if headless:
        options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36')
    
    if USE_WEBDRIVER_MANAGER:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)
    else:
        driver = webdriver.Chrome(options=options)
    
    return driver

# =============================================================================
# CORE SCRAPING FUNCTION
# =============================================================================
def scrape_var_page_selenium(driver, url, year_label):
    """
    Scrape VAR statistics from a single ESPN article page.
    
    Args:
        driver: Active Selenium WebDriver instance
        url: ESPN article URL to scrape
        year_label: Season label (e.g., "2023/2024") for the output DataFrame
    
    Returns:
        pd.DataFrame: Team VAR statistics with columns:
            - team_name: Club name
            - net_score: Net VAR impact (+/- or 0)
            - year: Season label
            - Various stat columns (overturns, goals, penalties, etc.)
    
    Page Structure (as of 2024):
        ESPN articles use <h2> tags for team headers in format "Team Name +N"
        followed by <p> tags containing detailed statistics.
        
    Selector Strategy:
        ESPN has changed their HTML structure multiple times. We try selectors
        in order of likelihood, falling back to broader searches if needed.
    """
    print(f"  🌐 Opening: {url}")
    driver.get(url)
    
    # Wait for React to render the article content.
    # ESPN's React hydration typically completes within 5-8 seconds.
    # Using 10s as a safe buffer for slower connections/systems.
    time.sleep(10)
    
    # Parse the fully-rendered DOM with BeautifulSoup for easier navigation
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    # ---------------------------------------------------------------------
    # SELECTOR STRATEGY: Try multiple CSS selectors in order of specificity
    # ESPN's article structure has evolved over the years (2019-2024)
    # ---------------------------------------------------------------------
    team_headers = []
    
    # Current ESPN format (2023+): article content in div.article-body
    team_headers = soup.select('div.article-body h2')
    if not team_headers:
        # Older format: section-based layout
        team_headers = soup.select('section.article-body h2')
    
    # Fallback: try generic article containers or all h2s
    if not team_headers:
        team_headers = soup.select('article h2') or soup.select('.Story__Body h2') or soup.find_all('h2')
    
    teams = []
    net_scores = []
    
    # ---------------------------------------------------------------------
    # TEAM NAME EXTRACTION with Regex Pattern Matching
    # ---------------------------------------------------------------------
    # Pattern: \s([+-]\d+|0)$ matches:
    #   - "Liverpool +4" ✓
    #   - "Manchester United 0" ✓  
    #   - "West Ham -3" ✓
    #   - "Ligue 1" ✗ (would match with [+-]?\d+ but not with our pattern)
    #   - "ESPN" ✗
    # ---------------------------------------------------------------------
    for h2 in team_headers:
        text = h2.text.strip()
        if text and re.search(r'\s([+-]\d+|0)$', text):
            split = text.rsplit(' ', 1)
            team = split[0].strip()
            score = split[1].strip()
            # Deduplicate: ESPN's nested DOM can produce duplicate h2 matches
            if team not in teams:
                teams.append(team)
                net_scores.append(score)
    
    # EPL has exactly 20 teams per season - cap to avoid edge cases
    teams = teams[:20]
    net_scores = net_scores[:20]
    
    print(f"  📊 Found {len(teams)} teams")
    
    # ---------------------------------------------------------------------
    # DETAILED STATISTICS EXTRACTION
    # Each team entry has a <p> tag with stats in format:
    #   "Overturns: 5\nRejected overturns: 2\nLeading to goals for: 1..."
    # ---------------------------------------------------------------------
    stats_blocks = soup.select('section.article-body p') or soup.select('div.article-body p')
    stat_texts = [p.text for p in stats_blocks if 'Overturns:' in p.text or 'Overturns' in p.text]
    
    # Handle mismatched team/stats counts (can occur if page structure varies)
    if len(teams) != len(stat_texts) and len(stat_texts) > 0:
        min_len = min(len(teams), len(stat_texts))
        teams = teams[:min_len]
        net_scores = net_scores[:min_len]
        stat_texts = stat_texts[:min_len]
    
    if len(teams) == 0:
        print(f"  ⚠️ No teams found for {year_label}")
        return pd.DataFrame()
    
    # Build initial DataFrame
    df = pd.DataFrame({
        'team_name': teams,
        'net_score': net_scores,
        'stats_combined': stat_texts if stat_texts else [''] * len(teams)
    })
    
    # ---------------------------------------------------------------------
    # COLUMN MAPPING: ESPN stat labels → DataFrame column names
    # The tuple format is (column_name, espn_label)
    # Note: "Penalties for / against" appears once but contains both values
    # ---------------------------------------------------------------------
    stats_col_mapping = [
        ('overturns_total', 'Overturns'),
        ('overturns_rejected', 'Rejected overturns'),
        ('leading_to_goals_for', 'Leading to goals for'),
        ('leading_to_goals_against', 'Leading to goals against'),
        ('disallowed_goals_for', 'Disallowed goals for'),
        ('disallowed_goals_against', 'Disallowed goals against'),
        ('net_goal_score', 'Net goal score'),
        ('subj_decisions_for', 'Subjective decisions for'),
        ('subj_decisions_against', 'Subjective decisions against'),
        ('net_subjective_score', 'Net subjective score'),
        ('penalties_for', 'Penalties for / against'),
        ('penalties_against', 'Penalties for / against')
    ]
    
    # Initialize stat columns with default value
    for col, _ in stats_col_mapping:
        df[col] = 0
    
    # Parse the combined stats text and populate individual columns
    for i, row in df.iterrows():
        if row['stats_combined']:
            lines = row['stats_combined'].split('\n')
            for line in lines:
                if ': ' in line:
                    parts = line.split(': ', 1)
                    key = parts[0].strip()
                    val = parts[1].strip() if len(parts) > 1 else ''
                    for colname, label in stats_col_mapping:
                        if label == key:
                            df.at[i, colname] = val
    
    # ---------------------------------------------------------------------
    # PENALTY DATA: Split "X / Y" format into separate columns
    # ESPN reports penalties as "Penalties for / against: 3 / 2"
    # ---------------------------------------------------------------------
    df['penalties_for'] = df['penalties_for'].apply(
        lambda x: x.split(' / ')[0] if isinstance(x, str) and ' / ' in x else x
    )
    df['penalties_against'] = df['penalties_against'].apply(
        lambda x: x.split(' / ')[1] if isinstance(x, str) and ' / ' in x else x
    )
    
    # Add season identifier and clean up temp columns
    df['year'] = year_label
    if 'stats_combined' in df.columns:
        df.drop(columns=['stats_combined'], inplace=True)
    
    return df

# =============================================================================
# DATA SOURCE CONFIGURATION
# =============================================================================
# Each tuple contains (URL, season_label)
# Add new seasons here when ESPN publishes end-of-season VAR analysis articles
# Note: 2025/2026 not yet available - articles typically published in May after season ends
espn_var_pages = [
    ("https://www.espn.com/soccer/story/_/id/37575919/how-var-decisions-affected-every-premier-league-club", "2019/2020"),
    ("https://www.espn.co.uk/football/story/_/id/37587139/how-var-decisions-affected-every-premier-league-club-2020-21", "2020/2021"),
    ("https://www.espn.co.uk/football/story/_/id/37619801/how-var-decisions-affected-every-premier-league-club-2021-22", "2021/2022"),
    ("https://www.espn.co.uk/football/story/_/id/37631044/how-var-decisions-affected-every-premier-league-club-2022-23", "2022/2023"),
    ("https://www.espn.com/soccer/story/_/id/38196464/how-var-decisions-affect-premier-league-club-2023-24", "2023/2024"),
    ("https://www.espn.co.uk/football/story/_/id/40894476/how-var-decisions-affect-premier-league-club-2024-25", "2024/2025")
]

# =============================================================================
# MAIN EXECUTION LOOP
# =============================================================================
# Strategy: Create a fresh WebDriver for each page to avoid session state issues.
# This is slower (~10s per page) but significantly more reliable than reusing drivers.
print("🚀 Starting VAR data scraping...")
all_dfs = []

for url, season in espn_var_pages:
    print(f"\n📆 Scraping: {season}")
    driver = init_driver(headless=True)
    try:
        df = scrape_var_page_selenium(driver, url, season)
        if not df.empty:
            all_dfs.append(df)
            print(f"  ✅ Got {len(df)} teams for {season}")
    except Exception as e:
        # Log but don't fail - continue with remaining seasons
        print(f"  ⚠️ Error scraping {season}: {e}")
    finally:
        # Always close the driver to free system resources
        driver.quit()
    # Rate limiting: be respectful to ESPN's servers
    time.sleep(2)

print("\n🔒 All scraping complete")

# =============================================================================
# OUTPUT: Combine all seasons and save to CSV
# =============================================================================
if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    
    # Export to CSV for downstream analysis
    output_file = "data/var_decisions_all_seasons.csv"
    df_all.to_csv(output_file, index=False)
    
    print(f"\n✅ Data saved to {output_file} ({len(df_all)} total rows)")
    print(f"   Seasons covered: {df_all['year'].nunique()}")
    print(f"   Teams per season: {len(df_all) // df_all['year'].nunique()}")
    
    # Preview the data
    display(df_all.head(10))
else:
    print("\n⚠️ No data was scraped. Possible causes:")
    print("   - ESPN changed their page structure")
    print("   - Network/timeout issues")
    print("   - ChromeDriver version mismatch")


🚀 Starting VAR data scraping...

📆 Scraping: 2019/2020
  🌐 Opening: https://www.espn.com/soccer/story/_/id/37575919/how-var-decisions-affected-every-premier-league-club
  📊 Found 20 teams
  ✅ Got 20 teams for 2019/2020


/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '12' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85


📆 Scraping: 2020/2021
  🌐 Opening: https://www.espn.co.uk/football/story/_/id/37587139/how-var-decisions-affected-every-premier-league-club-2020-21
  📊 Found 20 teams
  ✅ Got 20 teams for 2020/2021


/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85/


📆 Scraping: 2021/2022
  🌐 Opening: https://www.espn.co.uk/football/story/_/id/37619801/how-var-decisions-affected-every-premier-league-club-2021-22
  📊 Found 20 teams
  ✅ Got 20 teams for 2021/2022


...

/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85/


📆 Scraping: 2024/2025
  🌐 Opening: https://www.espn.co.uk/football/story/_/id/40894476/how-var-decisions-affect-premier-league-club-2024-25
  📊 Found 20 teams
  ✅ Got 20 teams for 2024/2025


/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '17' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85/93zmn9213r7g80k47fp3zdn80000gn/T/ipykernel_67517/820414534.py:234: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, colname] = val
/var/folders/85


🔒 All scraping complete

✅ Data saved to data/var_decisions_all_seasons.csv (120 total rows)
   Seasons covered: 6
   Teams per season: 20


,team_name,net_score,overturns_total,overturns_rejected,leading_to_goals_for,leading_to_goals_against,disallowed_goals_for,disallowed_goals_against,net_goal_score,subj_decisions_for,subj_decisions_against,net_subjective_score,penalties_for,penalties_against,year
0,Brighton & Hove Albion,+8,12,0,2,0,2,7,+7,2,0,+2,0,0,2019/2020
1,Manchester United,+7,13,0,1,2,0,7,+6,6,2,+4,0,0,2019/2020
2,Crystal Palace,+4,12,0,3,0,4,1,0,6,2,+4,0,0,2019/2020
3,Burnley,+3,11,0,2,1,3,4,+2,4,2,+2,0,0,2019/2020
4,Newcastle,+3,3,0,1,0,0,0,+1,2,0,2,0,0,2019/2020
5,Southampton,+3,13,0,0,1,0,7,+6,1,4,-3,0,0,2019/2020
6,Liverpool,+2,8,0,1,0,3,4,+2,1,1,0,0,0,2019/2020
7,Leicester City,+1,15,0,1,1,3,4,+1,3,3,0,0,0,2019/2020
8,Tottenham Hotspur,+1,15,0,1,1,4,6,+2,3,3,0,0,0,2019/2020
9,Manchester City,0,16,0,3,2,4,2,-1,4,4,0,0,0,2019/2020


In [2]:
"""
Single Season Analysis: 2019-2020
=================================
This cell demonstrates scraping a single season's VAR data along with the 
corresponding EPL league table for correlation analysis.

Data Sources:
    - VAR Data: ESPN article (2019-2020 was VAR's first full EPL season)
    - EPL Table: Hardcoded historical data (Eurosport's site structure changed)

Design Decision: Hardcoded EPL Standings
----------------------------------------
The original reference notebook (2021) scraped Eurosport for live standings.
Since then, Eurosport was rebranded to TNT Sports with a completely different
page structure. For historical analysis, hardcoding final standings is:
    1. More reliable (no dependency on external page structure)
    2. Accurate (historical data doesn't change)
    3. Faster (no network requests needed)

For current season standings, consider using the football-data.org API instead.
"""

# =============================================================================
# IMPORTS (duplicated for cell independence - can run cells individually)
# =============================================================================
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

try:
    from webdriver_manager.chrome import ChromeDriverManager
    USE_WEBDRIVER_MANAGER = True
except ImportError:
    USE_WEBDRIVER_MANAGER = False

def init_driver(headless=True):
    """Initialize Chrome WebDriver - see Cell 2 for detailed documentation."""
    options = Options()
    if headless:
        options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36')
    
    if USE_WEBDRIVER_MANAGER:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)
    else:
        driver = webdriver.Chrome(options=options)
    
    return driver

# =============================================================================
# 2019-2020 VAR DATA (ESPN)
# =============================================================================
# Note: This uses a different URL format than the multi-season scraper.
# ESPN restructured their article URLs at some point.
VAR_PAGE_2019_2020 = "https://www.espn.com/soccer/english-premier-league/story/3929823/how-var-decisions-have-affected-every-premier-league-club"

print("🚀 Initializing Chrome WebDriver...")
driver = init_driver(headless=True)

try:
    print(f"🌐 Opening VAR page: {VAR_PAGE_2019_2020}")
    driver.get(VAR_PAGE_2019_2020)
    time.sleep(5)  # Wait for JavaScript to render
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    # Try multiple selectors for older ESPN format
    # The 2019-2020 page uses h2 for team names with net score
    team_headers = soup.select('div.article-body h2') or soup.select('section.article-body h2')
    
    teams = []
    net_scores = []
    stats_list = []
    
    for h2 in team_headers:
        text = h2.text.strip()
        # Match pattern: "Team Name +N", "Team Name -N", or "Team Name 0"
        # The stricter regex ([+-]\d+|0) avoids false positives like "Ligue 1"
        if text and re.search(r'\s([+-]\d+|0)$', text):
            split = text.rsplit(' ', 1)
            team = split[0].strip()
            score = split[1].strip()
            # Deduplicate: ESPN's nested DOM can produce duplicate h2 matches
            if team not in teams:
                teams.append(team)
                net_scores.append(score)
    
    # EPL has exactly 20 teams - cap to avoid edge cases
    teams = teams[:20]
    net_scores = net_scores[:20]
    
    # Get stats paragraphs
    stats_blocks = soup.select('div.article-body p') or soup.select('section.article-body p')
    stat_texts = [p.text for p in stats_blocks if 'Overturns:' in p.text or 'Overturns' in p.text]
    
    print(f"📊 Found {len(teams)} teams with VAR data")
    
    if teams:
        # Ensure stats array matches teams array length
        stats_padded = stat_texts[:len(teams)] if len(stat_texts) >= len(teams) else stat_texts + [''] * (len(teams) - len(stat_texts))
        
        df_var = pd.DataFrame({
            "Club": teams,
            "Net Score": net_scores,
            "Stats": stats_padded
        })
        print("✅ VAR Decisions 2019–2020")
        display(df_var)
    else:
        print("⚠️ No VAR data found - page structure may have changed")
        df_var = pd.DataFrame(columns=["Club", "Net Score", "Stats"])
    
    # =========================================================================
    # EPL 2019-2020 FINAL STANDINGS (Hardcoded Historical Data)
    # =========================================================================
    # Source: Official Premier League records
    # 
    # Why hardcoded?
    #   - Eurosport (original source) was rebranded to TNT Sports
    #   - Their page structure completely changed, breaking the original scraper
    #   - Historical final standings never change, so hardcoding is reliable
    #   - Eliminates network dependency for historical analysis
    #
    # Data includes: Position, Team, Played, Won, Drawn, Lost, GF, GA, GD, Points
    # Liverpool won the title with a record 99 points (their first in 30 years)
    # =========================================================================
    print("\n📊 Loading EPL 2019-2020 Final Standings (hardcoded historical data)")
    
    epl_2019_2020_data = [
        {"Position": 1, "Team": "Liverpool", "Played": 38, "Won": 32, "Drawn": 3, "Lost": 3, "GF": 85, "GA": 33, "GD": 52, "Points": 99},
        {"Position": 2, "Team": "Manchester City", "Played": 38, "Won": 26, "Drawn": 3, "Lost": 9, "GF": 102, "GA": 35, "GD": 67, "Points": 81},
        {"Position": 3, "Team": "Manchester United", "Played": 38, "Won": 18, "Drawn": 12, "Lost": 8, "GF": 66, "GA": 36, "GD": 30, "Points": 66},
        {"Position": 4, "Team": "Chelsea", "Played": 38, "Won": 20, "Drawn": 6, "Lost": 12, "GF": 69, "GA": 54, "GD": 15, "Points": 66},
        {"Position": 5, "Team": "Leicester City", "Played": 38, "Won": 18, "Drawn": 8, "Lost": 12, "GF": 67, "GA": 41, "GD": 26, "Points": 62},
        {"Position": 6, "Team": "Tottenham Hotspur", "Played": 38, "Won": 16, "Drawn": 11, "Lost": 11, "GF": 61, "GA": 47, "GD": 14, "Points": 59},
        {"Position": 7, "Team": "Wolves", "Played": 38, "Won": 15, "Drawn": 14, "Lost": 9, "GF": 51, "GA": 40, "GD": 11, "Points": 59},
        {"Position": 8, "Team": "Arsenal", "Played": 38, "Won": 14, "Drawn": 14, "Lost": 10, "GF": 56, "GA": 48, "GD": 8, "Points": 56},
        {"Position": 9, "Team": "Sheffield United", "Played": 38, "Won": 14, "Drawn": 12, "Lost": 12, "GF": 39, "GA": 39, "GD": 0, "Points": 54},
        {"Position": 10, "Team": "Burnley", "Played": 38, "Won": 15, "Drawn": 9, "Lost": 14, "GF": 43, "GA": 50, "GD": -7, "Points": 54},
        {"Position": 11, "Team": "Southampton", "Played": 38, "Won": 15, "Drawn": 7, "Lost": 16, "GF": 51, "GA": 60, "GD": -9, "Points": 52},
        {"Position": 12, "Team": "Everton", "Played": 38, "Won": 13, "Drawn": 10, "Lost": 15, "GF": 44, "GA": 56, "GD": -12, "Points": 49},
        {"Position": 13, "Team": "Newcastle United", "Played": 38, "Won": 11, "Drawn": 11, "Lost": 16, "GF": 38, "GA": 58, "GD": -20, "Points": 44},
        {"Position": 14, "Team": "Crystal Palace", "Played": 38, "Won": 11, "Drawn": 10, "Lost": 17, "GF": 31, "GA": 50, "GD": -19, "Points": 43},
        {"Position": 15, "Team": "Brighton & Hove Albion", "Played": 38, "Won": 9, "Drawn": 14, "Lost": 15, "GF": 39, "GA": 54, "GD": -15, "Points": 41},
        {"Position": 16, "Team": "West Ham United", "Played": 38, "Won": 10, "Drawn": 9, "Lost": 19, "GF": 49, "GA": 62, "GD": -13, "Points": 39},
        {"Position": 17, "Team": "Aston Villa", "Played": 38, "Won": 9, "Drawn": 8, "Lost": 21, "GF": 41, "GA": 67, "GD": -26, "Points": 35},
        {"Position": 18, "Team": "AFC Bournemouth", "Played": 38, "Won": 9, "Drawn": 7, "Lost": 22, "GF": 40, "GA": 65, "GD": -25, "Points": 34},
        {"Position": 19, "Team": "Watford", "Played": 38, "Won": 8, "Drawn": 10, "Lost": 20, "GF": 36, "GA": 64, "GD": -28, "Points": 34},
        {"Position": 20, "Team": "Norwich City", "Played": 38, "Won": 5, "Drawn": 6, "Lost": 27, "GF": 26, "GA": 75, "GD": -49, "Points": 21},
    ]
    
    df_epl = pd.DataFrame(epl_2019_2020_data)
    df_epl["Year"] = "2019/2020"
    
    print(f"✅ EPL Table 2019–2020 ({len(df_epl)} teams)")
    display(df_epl)

finally:
    driver.quit()
    print("\n🔒 WebDriver closed")



🚀 Initializing Chrome WebDriver...
🌐 Opening VAR page: https://www.espn.com/soccer/english-premier-league/story/3929823/how-var-decisions-have-affected-every-premier-league-club
📊 Found 20 teams with VAR data
✅ VAR Decisions 2019–2020


,Club,Net Score,Stats
0,Brighton & Hove Albion,+8,Overturns: 12\nLeading to goals for: 2\nDisall...
1,Manchester United,+7,Overturns: 13\nLeading to goals for: 1\nDisall...
2,Crystal Palace,+4,Overturns: 12\nLeading to goals for: 3\nDisall...
3,Burnley,+3,Overturns: 11\nLeading to goals for: 2\nDisall...
4,Newcastle,+3,Overturns: 3\nLeading to goals for: 1\nDisallo...
5,Southampton,+3,Overturns: 13\nLeading to goals for: 0\nDisall...
6,Liverpool,+2,Overturns: 8\nLeading to goals for: 1\nDisallo...
7,Leicester City,+1,Overturns: 15\nLeading to goals for: 1\nDisall...
8,Tottenham Hotspur,+1,Overturns: 15\nLeading to goals for: 1\nDisall...
9,Manchester City,0,Overturns: 16\nLeading to goals for: 3\nDisall...



📊 Loading EPL 2019-2020 Final Standings (hardcoded historical data)
✅ EPL Table 2019–2020 (20 teams)


,Position,Team,Played,Won,Drawn,Lost,GF,GA,GD,Points,Year
0,1,Liverpool,38,32,3,3,85,33,52,99,2019/2020
1,2,Manchester City,38,26,3,9,102,35,67,81,2019/2020
2,3,Manchester United,38,18,12,8,66,36,30,66,2019/2020
3,4,Chelsea,38,20,6,12,69,54,15,66,2019/2020
4,5,Leicester City,38,18,8,12,67,41,26,62,2019/2020
5,6,Tottenham Hotspur,38,16,11,11,61,47,14,59,2019/2020
6,7,Wolves,38,15,14,9,51,40,11,59,2019/2020
7,8,Arsenal,38,14,14,10,56,48,8,56,2019/2020
8,9,Sheffield United,38,14,12,12,39,39,0,54,2019/2020
9,10,Burnley,38,15,9,14,43,50,-7,54,2019/2020



🔒 WebDriver closed


---

## Additional Data: FIFA World Rankings (Optional)

The cells below scrape historical FIFA World Rankings from Transfermarkt.
This data is **not required** for the EPL VAR analysis but may be useful for 
broader football analytics projects.

**Note:** These cells take a long time to run (30+ minutes) as they scrape 
rankings for every available date since 1992. Consider using only for one-time
data collection, then work from the saved CSV.

In [3]:
"""
FIFA World Rankings Historical Scraper (FAST Parallel Version)
===============================================================
Scrapes FIFA World Rankings from Transfermarkt using parallel requests.

Performance:
    - Original: 30-60 minutes (sequential)
    - This version: ~5-10 minutes (parallel with ThreadPoolExecutor)

Speed improvements:
    - Fetches multiple dates concurrently (5 at a time)
    - Fetches all pages for a date concurrently
    - Reduced sleep times (0.2s instead of 1-2s)
"""

import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
import time

# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_URL = "https://www.transfermarkt.us/statistik/weltrangliste/statistik"
HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}

# Concurrency settings - adjust if you get rate limited
MAX_DATE_WORKERS = 5      # How many dates to scrape in parallel
MAX_PAGE_WORKERS = 7      # How many pages per date in parallel
REQUEST_DELAY = 0.2       # Seconds between requests (per thread)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def get_all_date_options():
    """Fetch all available ranking dates from dropdown."""
    response = requests.get(BASE_URL, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(response.content, "html.parser")
    date_select = soup.find("select", {"name": "datum"})
    return [(opt["value"], opt.text.strip()) for opt in date_select.find_all("option")]

def get_total_pages(soup):
    """Extract total page count from pagination."""
    pagination = soup.select("div.pager ul.tm-pagination li a")
    page_numbers = [int(a.text) for a in pagination if a.text.strip().isdigit()]
    return max(page_numbers) if page_numbers else 1

def parse_page(html, date_value):
    """Parse a single page of rankings."""
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", class_="items")
    if not table:
        return []
    
    results = []
    for row in table.select("tbody > tr"):
        cols = row.find_all("td")
        if len(cols) < 4:
            continue
        
        rank = cols[0].get_text(strip=True).split(" ")[0]
        country = cols[1].img["alt"].strip() if cols[1].img else cols[1].get_text(strip=True)
        confed = cols[2].get_text(strip=True)
        points = cols[3].get_text(strip=True)
        
        try:
            results.append({
                "rank": int(rank),
                "country": country,
                "confederation": confed,
                "points": int(points) if points.isdigit() else 0,
                "date": date_value
            })
        except ValueError:
            continue
    
    return results

def fetch_page(url, date_value):
    """Fetch and parse a single page."""
    time.sleep(REQUEST_DELAY)
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        return parse_page(response.text, date_value)
    except Exception as e:
        print(f"  ⚠️ Error fetching {url}: {e}")
        return []

def scrape_date_parallel(date_value):
    """Scrape all pages for a date using parallel requests."""
    base_url = f"https://www.transfermarkt.us/statistik/weltrangliste/statistik/stat/ajax/yw1/datum/{date_value}/plus/0/galerie/0"
    
    # Get first page to determine total pages
    response = requests.get(base_url, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")
    total_pages = get_total_pages(soup)
    
    # Parse first page
    all_results = parse_page(response.text, date_value)
    
    if total_pages > 1:
        # Fetch remaining pages in parallel
        urls = [(f"{base_url}/page/{p}", date_value) for p in range(2, total_pages + 1)]
        
        with ThreadPoolExecutor(max_workers=MAX_PAGE_WORKERS) as executor:
            futures = [executor.submit(fetch_page, url, dv) for url, dv in urls]
            for future in as_completed(futures):
                all_results.extend(future.result())
    
    return all_results

# =============================================================================
# MAIN EXECUTION (Parallel)
# =============================================================================
print("🚀 Starting FAST parallel scraping...")
start_time = time.time()

dates = get_all_date_options()
print(f"📅 Found {len(dates)} dates to scrape")

# TIP: Uncomment to test with fewer dates
# dates = dates[:5]

all_entries = []
completed = 0

with ThreadPoolExecutor(max_workers=MAX_DATE_WORKERS) as executor:
    future_to_date = {executor.submit(scrape_date_parallel, dv): (dv, dt) for dv, dt in dates}
    
    for future in as_completed(future_to_date):
        date_value, date_text = future_to_date[future]
        completed += 1
        try:
            data = future.result()
            all_entries.extend(data)
            print(f"✅ [{completed}/{len(dates)}] {date_text}: {len(data)} countries")
        except Exception as e:
            print(f"❌ [{completed}/{len(dates)}] {date_text}: Error - {e}")

elapsed = time.time() - start_time
print(f"\n✅ Done in {elapsed/60:.1f} minutes! Scraped {len(all_entries)} entries.")

# =============================================================================
# EXPORT TO CSV
# =============================================================================
output_file = "fifa_world_rankings_full.csv"
with open(output_file, "w", newline='', encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["rank", "country", "confederation", "points", "date"])
    writer.writeheader()
    writer.writerows(all_entries)
    
print(f"💾 Saved to {output_file}")




🚀 Starting FAST parallel scraping...
📅 Found 365 dates to scrape
✅ [1/365] Dec 31, 1992: 162 countries
✅ [2/365] Nov 19, 1993: 166 countries
✅ [3/365] Dec 23, 1993: 166 countries
✅ [4/365] Feb 15, 1994: 165 countries
✅ [5/365] Aug 8, 1993: 165 countries
✅ [6/365] Oct 22, 1993: 165 countries
✅ [7/365] Sep 23, 1993: 165 countries
✅ [8/365] Mar 15, 1994: 167 countries
✅ [9/365] May 17, 1994: 169 countries
✅ [10/365] Apr 19, 1994: 167 countries
✅ [11/365] Jun 14, 1994: 171 countries
✅ [12/365] Jul 21, 1994: 171 countries
✅ [13/365] Sep 13, 1994: 171 countries
✅ [14/365] Oct 25, 1994: 173 countries
✅ [15/365] Dec 20, 1994: 178 countries
✅ [16/365] Nov 22, 1994: 176 countries
✅ [17/365] Feb 20, 1995: 177 countries
✅ [18/365] Apr 19, 1995: 177 countries
✅ [19/365] May 16, 1995: 177 countries
✅ [20/365] Jul 25, 1995: 178 countries
✅ [21/365] Jun 13, 1995: 177 countries
✅ [22/365] Aug 22, 1995: 178 countries
✅ [23/365] Sep 19, 1995: 171 countries
✅ [24/365] Dec 19, 1995: 181 countries
✅ [25/365

In [4]:
"""
Google Colab File Download Helper
==================================
This cell is only needed when running in Google Colab.
It triggers a browser download of the generated CSV file.

Note: This will fail in Jupyter/VS Code - skip this cell if not using Colab.
"""

from google.colab import files

filename = "fifa_world_rankings_full.csv"
with open(filename, "w", newline='', encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["rank", "country", "confederation", "points", "date"])
    writer.writeheader()
    writer.writerows(all_entries)

print("💾 Saved to", filename)
files.download(filename)  # Triggers browser download in Colab


ModuleNotFoundError: No module named 'google'

### Google Colab: Prevent Disconnects

**Note:** Only relevant if running this notebook in Google Colab.

Colab automatically disconnects after ~90 minutes of inactivity. Since the FIFA rankings
scraper takes 30-60 minutes to run, this usually isn't an issue. However, if you're running
multiple long operations or stepping away, use this JavaScript hack:

**Steps:**
1. Open browser console: `Ctrl+Shift+J` (Windows/Linux) or `Cmd+Opt+J` (Mac)
2. Paste the following code:

```javascript
function ClickConnect(){
  console.log("⏳ Preventing disconnect...");
  document.querySelector("colab-toolbar-button#connect").click();
}
setInterval(ClickConnect, 60000); // Clicks "Connect" button every 60 seconds
```

**Alternative:** Use Colab Pro for longer session timeouts.

In [ ]:
"""
FIFA World Rankings Historical Scraper (Async Version)
======================================================
Same data as Cell 5, but uses async/await for parallel page fetching within each date.

Performance Improvement:
    - Synchronous (Cell 5): Sequential page fetches, ~30-60 minutes
    - Async (this cell): Parallel page fetches per date, ~15-30 minutes

Dependencies:
    - httpx: Async HTTP client (pip install httpx)
    - nest_asyncio: Enables async in Jupyter notebooks (pip install nest_asyncio)

Architecture:
    - For each date, all 7 pages are fetched concurrently using asyncio.gather()
    - Between dates, we still wait 2 seconds to be respectful to the server
    - This gives us the speed benefit of async while not overwhelming the server
"""

import nest_asyncio
import asyncio
import httpx
from bs4 import BeautifulSoup
import csv

# Enable nested event loops (required for Jupyter notebooks)
nest_asyncio.apply()

# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_URL = "https://www.transfermarkt.us/statistik/weltrangliste/statistik"
HEADERS = {"User-Agent": "Mozilla/5.0"}


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def get_all_date_options():
    """Fetch available dates synchronously (only called once at start)."""
    response = httpx.get(BASE_URL, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")
    date_select = soup.find("select", {"name": "datum"})
    return [(opt["value"], opt.text.strip()) for opt in date_select.find_all("option")]


async def get_total_pages(client, url):
    """Async fetch to determine pagination count."""
    response = await client.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")
    pagination = soup.select("div.pager ul.tm-pagination li a")
    page_numbers = [int(a.text) for a in pagination if a.text.isdigit()]
    return max(page_numbers) if page_numbers else 1


def parse_table(html, date_value):
    """Parse ranking table HTML into list of dicts (pure function, no I/O)."""
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", class_="items")
    if not table:
        return []

    results = []
    rows = table.select("tbody > tr")
    for row in rows:
        cols = row.find_all("td")
        if len(cols) < 4:
            continue

        rank = cols[0].get_text(strip=True).split(" ")[0]
        country = cols[1].img["alt"].strip() if cols[1].img else cols[1].get_text(strip=True)
        confed = cols[2].get_text(strip=True)
        points = cols[3].get_text(strip=True)

        try:
            points_int = int(points)
        except ValueError:
            points_int = 0

        results.append({
            "rank": int(rank),
            "country": country,
            "confederation": confed,
            "points": points_int,
            "date": date_value
        })

    return results


async def fetch_page(client, url, date_value):
    """Async fetch a single page and parse it."""
    try:
        response = await client.get(url, headers=HEADERS)
        response.raise_for_status()
        return parse_table(response.text, date_value)
    except Exception as e:
        print(f"❌ Failed to fetch {url}: {e}")
        return []


async def scrape_all_pages_for_date(client, date_value):
    """
    Scrape all pages for a date CONCURRENTLY using asyncio.gather().
    This is where the async speed benefit comes from.
    """
    base_url = f"https://www.transfermarkt.us/statistik/weltrangliste/statistik/stat/ajax/yw1/datum/{date_value}/plus/0/galerie/0"
    
    print(f"⏳ Getting total pages for {date_value}...")
    total_pages = await get_total_pages(client, base_url)
    print(f"📄 Total pages for {date_value}: {total_pages}")

    # Build list of concurrent tasks for all pages
    tasks = []
    for page in range(1, total_pages + 1):
        if page == 1:
            url = base_url
        else:
            url = f"{base_url}/page/{page}"
        print(f"🔎 Scheduling page {page}...")
        tasks.append(fetch_page(client, url, date_value))

    # Execute all page fetches concurrently
    results = await asyncio.gather(*tasks)
    
    # Flatten list of lists into single list
    return [item for sublist in results for item in sublist]


# =============================================================================
# MAIN ASYNC EXECUTION
# =============================================================================
async def main():
    all_entries = []
    dates = get_all_date_options()
    
    # TIP: Uncomment to test with just 2 dates
    # dates = dates[:2]

    # httpx.AsyncClient manages connection pooling for efficiency
    async with httpx.AsyncClient(timeout=15.0) as client:
        for date_value, date_text in dates:
            print(f"\n📅 Scraping {date_text} ({date_value})...")
            try:
                results = await scrape_all_pages_for_date(client, date_value)
                all_entries.extend(results)
            except Exception as e:
                print(f"❌ Error scraping {date_value}: {e}")
            
            # Rate limiting between dates (pages within a date are concurrent)
            await asyncio.sleep(2)

    # Export results
    output_file = "fifa_world_rankings_full.csv"
    with open(output_file, "w", newline='', encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["rank", "country", "confederation", "points", "date"])
        writer.writeheader()
        writer.writerows(all_entries)

    print(f"\n✅ Done. Scraped total: {len(all_entries)} entries.")
    print(f"💾 Saved to {output_file}")


# Run the async main function
await main()


ModuleNotFoundError: No module named 'httpx'